# 知识蒸馏教程 (Knowledge Distillation Tutorial)

本教程详细介绍知识蒸馏技术，包括：

1. **蒸馏基础**: 软标签和温度参数
2. **响应蒸馏**: 匹配输出分布
3. **特征蒸馏**: 匹配中间层特征
4. **关系蒸馏**: 保持样本间关系

---

## 什么是知识蒸馏？

知识蒸馏将大型教师模型的"知识"迁移到小型学生模型：

```
教师模型 (Teacher)           学生模型 (Student)
┌─────────────────┐         ┌─────────────┐
│  大型预训练模型  │   →     │  轻量级模型  │
│  参数: 数十亿    │  蒸馏    │  参数: 数百万 │
│  精度: 高       │         │  精度: 接近   │
└─────────────────┘         └─────────────┘
```

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch 版本: {torch.__version__}")

## 1. 软标签与温度参数

### 硬标签 vs 软标签

**硬标签**: one-hot 编码，如 [0, 0, 1, 0, 0]

**软标签**: 概率分布，如 [0.05, 0.1, 0.7, 0.1, 0.05]

软标签包含更多信息：类别之间的相似性关系。

In [ ]:
from distillation import soft_cross_entropy

# 模拟教师模型输出 (logits)
teacher_logits = torch.tensor([[2.0, 1.0, 0.5, -1.0, -2.0]])

# 不同温度下的软标签
temperatures = [1.0, 2.0, 4.0, 8.0]

fig, axes = plt.subplots(1, 4, figsize=(16, 3))

for ax, T in zip(axes, temperatures):
    soft_labels = F.softmax(teacher_logits / T, dim=-1).squeeze().numpy()
    ax.bar(range(5), soft_labels, color='steelblue', alpha=0.7)
    ax.set_title(f'温度 T = {T}')
    ax.set_xlabel('类别')
    ax.set_ylabel('概率')
    ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

print("观察: 温度越高，分布越平滑，暴露更多类间关系信息")

In [ ]:
# 温度参数的数学解释
print("温度参数的作用:")
print("="*50)
print(f"{'温度':<10} {'最大概率':<15} {'熵':<15}")
print("-"*50)

for T in [0.5, 1.0, 2.0, 4.0, 8.0]:
    probs = F.softmax(teacher_logits / T, dim=-1).squeeze()
    max_prob = probs.max().item()
    entropy = -(probs * probs.log()).sum().item()
    print(f"{T:<10} {max_prob:<15.4f} {entropy:<15.4f}")

print("\n熵越高，分布越均匀，包含更多"暗知识"")

## 2. 定义教师和学生模型

In [ ]:
# 教师模型 (较大)
class TeacherModel(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=512, num_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.fc4 = nn.Linear(hidden_dim // 2, num_classes)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
    
    def forward(self, x):
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.dropout(self.relu(self.fc2(x)))
        x = self.relu(self.fc3(x))
        return self.fc4(x)

# 学生模型 (较小)
class StudentModel(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=128, num_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_classes)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.relu(self.fc1(x))
        return self.fc2(x)

# 创建模型
teacher = TeacherModel()
student = StudentModel()

# 统计参数
teacher_params = sum(p.numel() for p in teacher.parameters())
student_params = sum(p.numel() for p in student.parameters())

print(f"教师模型参数量: {teacher_params:,}")
print(f"学生模型参数量: {student_params:,}")
print(f"压缩比: {teacher_params / student_params:.1f}x")

## 3. 响应蒸馏 (Response-based Distillation)

最基本的蒸馏方法：让学生模型学习教师模型的输出分布。

**损失函数**:
$$L = \alpha \cdot L_{soft} + (1 - \alpha) \cdot L_{hard}$$

其中:
- $L_{soft}$: KL 散度 (软标签)
- $L_{hard}$: 交叉熵 (硬标签)
- $\alpha$: 平衡系数

In [ ]:
from distillation import DistillationLoss

# 创建蒸馏损失
distill_loss = DistillationLoss(temperature=4.0, alpha=0.7)

# 模拟数据
x = torch.randn(32, 784)
labels = torch.randint(0, 10, (32,))

# 前向传播
teacher.eval()
with torch.no_grad():
    teacher_logits = teacher(x)

student.train()
student_logits = student(x)

# 计算损失
total_loss, loss_dict = distill_loss(student_logits, teacher_logits, labels)

print("蒸馏损失分解:")
print(f"  软标签损失 (KL): {loss_dict['soft_loss']:.4f}")
print(f"  硬标签损失 (CE): {loss_dict['hard_loss']:.4f}")
print(f"  总损失: {loss_dict['total_loss']:.4f}")

In [ ]:
# 完整的蒸馏训练
from distillation import KnowledgeDistiller, DistillationConfig

# 重新创建模型
teacher = TeacherModel()
student = StudentModel()

# 模拟"预训练"教师模型
print("模拟训练教师模型...")
teacher_optimizer = torch.optim.Adam(teacher.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

train_data = [(torch.randn(64, 784), torch.randint(0, 10, (64,))) for _ in range(100)]

teacher.train()
for epoch in range(5):
    total_loss = 0
    for x_batch, y_batch in train_data:
        teacher_optimizer.zero_grad()
        output = teacher(x_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        teacher_optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 2 == 0:
        print(f"  Epoch {epoch+1}: Loss = {total_loss/len(train_data):.4f}")

print("教师模型训练完成!")

In [ ]:
# 使用 KnowledgeDistiller 进行蒸馏
config = DistillationConfig(
    temperature=4.0,
    alpha=0.7,
    learning_rate=1e-3,
    num_epochs=10
)

distiller = KnowledgeDistiller(teacher, student, config)

print("\n开始知识蒸馏...")
trained_student = distiller.train(
    train_data,
    num_epochs=10,
    verbose=True
)

print("\n蒸馏完成!")

In [ ]:
# 可视化训练历史
history = distiller.get_history()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

epochs = range(1, len(history) + 1)

axes[0].plot(epochs, [h['total'] for h in history], 'b-', label='总损失')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('损失')
axes[0].set_title('训练损失曲线')
axes[0].legend()

axes[1].plot(epochs, [h['soft'] for h in history], 'g-', label='软标签损失')
axes[1].plot(epochs, [h['hard'] for h in history], 'r-', label='硬标签损失')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('损失')
axes[1].set_title('损失分解')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. 特征蒸馏 (Feature-based Distillation)

除了输出层，还可以匹配中间层特征。

In [ ]:
from distillation import FeatureDistillation, AttentionTransfer

# 特征蒸馏模块
feature_distiller = FeatureDistillation(
    student_channels=128,  # 学生特征维度
    teacher_channels=512,  # 教师特征维度
    use_projector=True     # 使用投影层对齐维度
)

# 模拟特征
student_features = torch.randn(32, 128)
teacher_features = torch.randn(32, 512)

# 计算特征蒸馏损失
feature_loss = feature_distiller(student_features, teacher_features)
print(f"特征蒸馏损失: {feature_loss.item():.4f}")

In [ ]:
# 注意力迁移 (用于 CNN)
attention_transfer = AttentionTransfer(p=2)

# 模拟 CNN 特征图
student_fmap = torch.randn(8, 32, 7, 7)  # [batch, channels, H, W]
teacher_fmap = torch.randn(8, 64, 7, 7)

# 计算注意力图
student_attention = attention_transfer.attention_map(student_fmap)
teacher_attention = attention_transfer.attention_map(teacher_fmap)

print(f"学生注意力图形状: {student_attention.shape}")
print(f"教师注意力图形状: {teacher_attention.shape}")

# 注意力迁移损失
at_loss = attention_transfer(student_fmap, teacher_fmap)
print(f"\n注意力迁移损失: {at_loss.item():.4f}")

## 5. 关系蒸馏 (Relation-based Distillation)

保持样本之间的关系结构。

In [ ]:
from distillation import RelationDistillation, DistanceWiseDistillation

# 关系蒸馏
relation_distiller = RelationDistillation(distance_type="cosine")

# 模拟特征
student_features = torch.randn(16, 128)
teacher_features = torch.randn(16, 512)

# 计算相似度矩阵
student_sim = relation_distiller.compute_similarity_matrix(student_features)
teacher_sim = relation_distiller.compute_similarity_matrix(teacher_features)

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

im1 = axes[0].imshow(student_sim.detach().numpy(), cmap='coolwarm', vmin=-1, vmax=1)
axes[0].set_title('学生模型相似度矩阵')
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(teacher_sim.detach().numpy(), cmap='coolwarm', vmin=-1, vmax=1)
axes[1].set_title('教师模型相似度矩阵')
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()

# 关系蒸馏损失
relation_loss = relation_distiller(student_features, teacher_features)
print(f"关系蒸馏损失: {relation_loss.item():.4f}")

## 6. 蒸馏 vs 直接训练

比较蒸馏学生和直接训练学生的效果。

In [ ]:
# 直接训练学生模型 (不使用蒸馏)
student_direct = StudentModel()
optimizer_direct = torch.optim.Adam(student_direct.parameters(), lr=1e-3)

print("直接训练学生模型 (无蒸馏)...")
student_direct.train()
direct_losses = []

for epoch in range(10):
    total_loss = 0
    for x_batch, y_batch in train_data:
        optimizer_direct.zero_grad()
        output = student_direct(x_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer_direct.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_data)
    direct_losses.append(avg_loss)
    if (epoch + 1) % 5 == 0:
        print(f"  Epoch {epoch+1}: Loss = {avg_loss:.4f}")

print("直接训练完成!")

In [ ]:
# 比较模型输出
test_data = torch.randn(100, 784)

teacher.eval()
trained_student.eval()
student_direct.eval()

with torch.no_grad():
    teacher_out = teacher(test_data)
    distilled_out = trained_student(test_data)
    direct_out = student_direct(test_data)

# 计算与教师的差异
distilled_diff = (teacher_out - distilled_out).abs().mean().item()
direct_diff = (teacher_out - direct_out).abs().mean().item()

print("与教师模型输出的差异:")
print(f"  蒸馏学生: {distilled_diff:.4f}")
print(f"  直接训练学生: {direct_diff:.4f}")

# 预测一致性
teacher_pred = teacher_out.argmax(dim=1)
distilled_pred = distilled_out.argmax(dim=1)
direct_pred = direct_out.argmax(dim=1)

distilled_match = (teacher_pred == distilled_pred).float().mean().item()
direct_match = (teacher_pred == direct_pred).float().mean().item()

print(f"\n与教师预测一致率:")
print(f"  蒸馏学生: {distilled_match * 100:.1f}%")
print(f"  直接训练学生: {direct_match * 100:.1f}%")

## 7. 温度和 Alpha 参数调优

In [ ]:
# 测试不同温度
temperatures = [1.0, 2.0, 4.0, 8.0, 16.0]
temp_results = []

for T in temperatures:
    student_temp = StudentModel()
    config_temp = DistillationConfig(temperature=T, alpha=0.7, num_epochs=5)
    distiller_temp = KnowledgeDistiller(teacher, student_temp, config_temp)
    
    distiller_temp.train(train_data, num_epochs=5, verbose=False)
    
    # 评估
    student_temp.eval()
    with torch.no_grad():
        out = student_temp(test_data)
    match = (teacher_pred == out.argmax(dim=1)).float().mean().item()
    temp_results.append(match)

# 可视化
plt.figure(figsize=(8, 4))
plt.plot(temperatures, [r * 100 for r in temp_results], 'bo-')
plt.xlabel('温度 T')
plt.ylabel('与教师预测一致率 (%)')
plt.title('温度参数对蒸馏效果的影响')
plt.grid(True, alpha=0.3)
plt.show()

## 总结

本教程介绍了知识蒸馏的核心概念：

1. **软标签**: 包含类间关系的"暗知识"
2. **温度参数**: 控制软标签的平滑程度
3. **响应蒸馏**: 匹配输出分布
4. **特征蒸馏**: 匹配中间层特征
5. **关系蒸馏**: 保持样本间关系

### 选择建议

- **简单任务**: 响应蒸馏
- **复杂任务**: 响应 + 特征蒸馏
- **小数据集**: 关系蒸馏